In [1]:
import numpy as np

In [10]:
class Car:
    def __init__(self, model, brand, year):
        self.model = model
        self.brand = brand
        self.year = year
        self.is_running = True
    
    def start_engine(self):
        self.is_running = True
        print (f"เครื่องของ {self.model} {self.brand} {self.year} กำลังสตาร์ท!")
    
    def stop_engine(self):
        self.is_runnig = False
        print (f"เครื่องของ {self.model} {self.brand} {self.year} กำลังดับ!")
    
car1 = Car("BMW", "model 3", 2023)
car2 = Car("Benz", "CLA3", 2024)
    
car1.start_engine()
car2.stop_engine()
    
print (f"รถที่วัสดุดีที่สุด {car1.model} {car1.brand} {car1.year}")
        

เครื่องของ BMW model 3 2023 กำลังสตาร์ท!
เครื่องของ Benz CLA3 2024 กำลังดับ!
รถที่วัสดุดีที่สุด BMW model 3 2023


## สาย Finance ที่ใช้งาน Genarate จาก Claude

In [ ]:
from decimal import Decimal
from datetime import datetime
from dataclasses import dataclass, field
from typing import List
import uuid

# ---- Data class เล็กๆ สำหรับ Transaction ----
class Transaction:
    def __init__(self, amount: Decimal, txn_type: str, description: str):
        self.id = str(uuid.uuid4())
        self.amount = amount
        self.type = txn_type
        self.timestamp = datetime.now()
        self.description = description

# ---- Base Class ----
class BankAccount:
    def __init__(self, account_number: str, owner_id: str):
        self._account_number = account_number
        self._owner_id = owner_id
        self._balance = Decimal("0.00")
        self._transactions: List[Transaction] = []
        self._is_frozen = False

    @property
    def balance(self):
        return self._balance

    @property
    def account_number(self):
        return self._account_number

    def deposit(self, amount: Decimal, description: str = "") -> Transaction:
        self._validate_not_frozen()
        self._validate_amount(amount)
        self._balance += amount
        return self._record_transaction(amount, "CREDIT", description)

    def withdraw(self, amount: Decimal, description: str = "") -> Transaction:
        self._validate_not_frozen()
        self._validate_amount(amount)
        self._validate_withdrawal(amount)
        self._balance -= amount
        return self._record_transaction(amount, "DEBIT", description)

    def _validate_withdrawal(self, amount: Decimal):
        raise NotImplementedError("Subclass must implement this")

    def _validate_not_frozen(self):
        if self._is_frozen:
            raise ValueError(f"Account {self._account_number} is frozen")

    def _validate_amount(self, amount: Decimal):
        if amount <= 0:
            raise ValueError("Amount must be positive")

    def _record_transaction(self, amount, txn_type, desc):
        txn = Transaction(amount, txn_type, desc)
        self._transactions.append(txn)
        return txn

# ---- Subclasses ----
class SavingsAccount(BankAccount):
    MAX_MONTHLY_WITHDRAWALS = 5

    def __init__(self, account_number: str, owner_id: str):
        super().__init__(account_number, owner_id)
        self._monthly_withdrawal_count = 0

    def _validate_withdrawal(self, amount: Decimal):
        if self._balance < amount:
            raise ValueError(f"Insufficient funds. Balance: {self._balance}, Requested: {amount}")
        if self._monthly_withdrawal_count >= self.MAX_MONTHLY_WITHDRAWALS:
            raise ValueError("Monthly withdrawal limit reached")

    def withdraw(self, amount: Decimal, description: str = "") -> Transaction:
        txn = super().withdraw(amount, description)
        self._monthly_withdrawal_count += 1
        return txn


class CheckingAccount(BankAccount):
    def __init__(self, account_number: str, owner_id: str, overdraft_limit: Decimal):
        super().__init__(account_number, owner_id)
        self._overdraft_limit = overdraft_limit

    def _validate_withdrawal(self, amount: Decimal):
        available = self._balance + self._overdraft_limit
        if amount > available:
            raise ValueError(f"Exceeds overdraft limit. Available: {available}")


class FixedDepositAccount(BankAccount):
    def __init__(self, account_number: str, owner_id: str, maturity_date: datetime):
        super().__init__(account_number, owner_id)
        self._maturity_date = maturity_date

    def _validate_withdrawal(self, amount: Decimal):
        if datetime.now() < self._maturity_date:
            raise ValueError(f"Cannot withdraw before maturity date: {self._maturity_date}")
        if self._balance < amount:
            raise ValueError("Insufficient funds")


# ---- Transfer Service ----
class TransferService:
    def transfer(self, from_account: BankAccount, to_account: BankAccount,
                 amount: Decimal, description: str = "") -> dict:
        try:
            from_account.withdraw(amount, f"Transfer to {to_account.account_number}: {description}")
            to_account.deposit(amount, f"Transfer from {from_account.account_number}: {description}")
            return {"status": "SUCCESS"}
        except ValueError as e:
            return {"status": "FAILED", "reason": str(e)}


# ---- Demo ----
print("=" * 50)
print("🏦 BANK SYSTEM DEMO")
print("=" * 50)

savings = SavingsAccount("SAV001", "USER1")
checking = CheckingAccount("CHK001", "USER2", overdraft_limit=Decimal("5000"))

savings.deposit(Decimal("10000"), "Initial deposit")
print(f"💰 Savings balance: {savings.balance}")

checking.deposit(Decimal("3000"), "Initial deposit")
print(f"💰 Checking balance: {checking.balance}")

service = TransferService()
result = service.transfer(savings, checking, Decimal("2000"), "Rent")
print(f"\n📤 Transfer result: {result['status']}")
print(f"   Savings after:  {savings.balance}")
print(f"   Checking after: {checking.balance}")

print("\n⚠️  ทดสอบถอนเงินเกิน:")
try:
    savings.withdraw(Decimal("999999"))
except ValueError as e:
    print(f"   Error: {e}")

print("\n⚠️  ทดสอบบัญชีฝากประจำ:")
fixed = FixedDepositAccount("FIX001", "USER3", maturity_date=datetime(2099, 1, 1))
fixed.deposit(Decimal("50000"), "Fixed deposit")
try:
    fixed.withdraw(Decimal("1000"))
except ValueError as e:
    print(f"   Error: {e}")